# Separate labelling step: does it beat the extraction's labels?

Today, direction and magnitude come from the extraction call, which lists and labels surprises at once. The new pipeline (deck slide 1) labels them in calls of their own, after classification, and adds management **sentiment**. This notebook tests that on the **train set only**; test events are never labelled or scored here.

1. Inspect the prompt and one labelled batch.
2. Label every train surprise (`labeling.run_labeling`, resumable; batches of 40 surprises, 8 workers).
3. Compare the new labels with the extraction's on the same surprises.
4. Build event features under both label sets, on the same events.
5. No-fit comparison: single scores' ΔR² on train, paired bootstrap.
6. Ridge cross-validation under both label sets on the same folds, paired on out-of-fold predictions.

**Decision rule.** Switch to the separate step only if it beats the extraction labels on pooled out-of-fold ΔR² with a paired bootstrap probability of at least 0.9. Otherwise keep the current pipeline, which costs one call less per event.

Labelling: `gpt-6-luna`, medium effort (`labeling.LABEL_MODEL`, `LABEL_EFFORT`). **The labeller sees only the surprises**: category, description, call evidence, expectation evidence and expectation source. No call facts, preview or option statistics, so a batch can mix surprises from different events. Only events whose every surprise got a label are compared.

In [6]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from ml import metrics, model, preprocess
from suprise_dim import data, dataset, features, grouping, labeling
from suprise_dim.prompts import LABEL_SYSTEM, build_label_message

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

split = data.load_split()
events_by_id = {e["event_id"]: e for e in data.load_eligible_events()}
tax = grouping.load_taxonomy("v2")
surprises = pd.read_csv(dataset.surprises_path("v2"))
train_items = surprises[surprises["event_id"].isin(split["train"])].sort_values(["event_id", "position"]).reset_index(drop=True)
RUN = "v2_train"  # data/surprise/labels_v2_train.jsonl
print(f"{train_items['event_id'].nunique()} train events | {len(train_items)} train surprises | {-(-len(train_items) // 40)} batches of 40")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
1076 train events | 8343 train surprises | 209 batches of 40


## 1. Inspect the prompt and one batch

In [7]:
batch = train_items.head(40)
print(LABEL_SYSTEM)
print("\n" + "=" * 80 + "\n")
print(build_label_message(batch)[:3000])

You label surprises from earnings releases.

You receive a numbered list of surprises, possibly from different companies. Each surprise is one way a company's reported results, guidance or commentary differed from what was expected before the release. Each comes with its category, a description, the earnings-call evidence, the pre-release expectation evidence, and where the expectation came from (the research preview, the call itself, or implicit). Judge each surprise on its own, from this information only.

For each surprise, give:
- direction: 1 if it is good news for shareholders relative to what was expected, -1 if bad news, 0 if it combines good and bad parts that roughly offset. null only if the information does not let you tell.
- magnitude: the size of the gap relative to the expectation, not how important the topic is. 1 small (a slight beat or miss, within normal noise), 2 moderate (clearly outside the expected range), 3 large (far outside it, or a reset of the outlook). null

In [8]:
# One live call on the first batch: new labels next to the extraction's.
new = pd.DataFrame(labeling.label_batch(batch))
batch[["item_id", "dimension", "direction", "magnitude", "description"]].merge(
    new.drop(columns="event_id"), on="item_id", suffixes=("_extraction", "_new"))

,item_id,dimension,direction_extraction,magnitude_extraction,description,direction_new,magnitude_new,sentiment,reason
0,ea_AAOI_Q2_2026#0,forward_guidance,negative,moderate,The preview looked for Q3 revenue of roughly $...,-1,2,-1,The Q3 midpoint is well below the preview rang...
1,ea_AAOI_Q2_2026#1,forward_guidance,negative,moderate,The preview's central expectation was that AOI...,-1,1,-1,“Around” $1.1 billion falls short of the previ...
2,ea_AAOI_Q2_2026#2,financing_capital_structure,negative,large,The preview flagged equity issuance as a finan...,-1,3,0,The $538.8 million equity raise is a substanti...
3,ea_AAOI_Q2_2026#3,forward_guidance,negative,moderate,The preview's margin math implied roughly 37% ...,-1,2,0,The Q4 margin target is below the preview-impl...
4,ea_AAOI_Q2_2026#4,forward_guidance,negative,moderate,The preview discussed laser and MOCVD constrai...,-1,2,-1,The unexpected memory shortage is expected to ...
5,ea_AAOI_Q2_2026#5,forward_guidance,positive,moderate,The preview expected Q2 CATV revenue of $75–80...,1,2,1,CATV beat the Q2 range and Q3 guidance signals...
6,ea_AAOI_Q2_2026#6,commercial_economics,positive,small,The preview did not anticipate a specific DOCS...,1,1,1,The unanticipated primary-vendor selection is ...
7,ea_AAOI_Q2_2026#7,operating_kpis,positive,moderate,The preview expected Q2 manufacturing capacity...,1,2,1,Capacity is already well above the preview’s Q...
8,ea_AAOI_Q2_2026#8,capital_allocation,mixed,large,AOI reported $565.5 million in Q2 capital inve...,0,3,0,The exceptionally large investment supports ca...
9,ea_AAOI_Q2_2026#9,bottom_line_earnings,mixed,moderate,"Q2 non-GAAP EPS was expected at $0.02–$0.03, b...",1,2,0,"EPS substantially exceeded the preview range, ..."


## 2. Label every train surprise

`N` counts surprises. Start with a small `N` as a smoke test, then `N = None` (8,343 surprises, about 209 calls of 40). Reruns resume and retry surprises the model skipped.

In [9]:
N = None  # None for every train surprise

summary = labeling.run_labeling(train_items.head(N) if N else train_items, RUN, batch_size=40, workers=8)
summary

Labelling 8343 surprises in 209 batches (0 already done) with gpt-6-luna...
[1/209] 40/40 labelled
[2/209] 40/40 labelled
[3/209] 40/40 labelled
[4/209] 40/40 labelled
[5/209] 40/40 labelled
[6/209] 40/40 labelled
[7/209] 40/40 labelled
[8/209] 40/40 labelled
[9/209] 40/40 labelled
[10/209] 40/40 labelled
[11/209] 40/40 labelled
[12/209] 40/40 labelled
[13/209] 40/40 labelled
[14/209] 40/40 labelled
[15/209] 40/40 labelled
[16/209] 40/40 labelled
[17/209] 40/40 labelled
[18/209] 40/40 labelled
[19/209] 40/40 labelled
[20/209] 40/40 labelled
[21/209] 40/40 labelled
[22/209] 40/40 labelled
[23/209] 40/40 labelled
[24/209] 40/40 labelled
[25/209] 40/40 labelled
[26/209] 40/40 labelled
[27/209] 40/40 labelled
[28/209] 40/40 labelled
[29/209] 40/40 labelled
[30/209] 40/40 labelled
[31/209] 40/40 labelled
[32/209] 40/40 labelled
[33/209] 40/40 labelled
[34/209] 40/40 labelled
[35/209] 40/40 labelled
[36/209] 40/40 labelled
[37/209] 40/40 labelled
[38/209] 40/40 labelled
[39/209] 40/40 labell

{'requested': 8343,
 'skipped': 0,
 'written': 8339,
 'missing': 4,
 'failed_batches': 0}

## 3. New labels vs the extraction's

In [14]:
labels = labeling.load_labels(RUN)
labelled_ids = labeling.complete_events(train_items, labels)
labelled = labeling.apply_labels(surprises, labels, labelled_ids)
agree = labeling.label_agreement(labelled)
print(f"{len(labels)} surprises labelled | {len(labelled_ids)} events complete | comparing {agree['n']} surprises | same direction {agree['direction_same']:.1%} | same magnitude {agree['magnitude_same']:.1%}")
display(agree["direction_table"])
display(agree["magnitude_table"])
agree["sentiment_by_direction"]

8339 surprises labelled | 1072 events complete | comparing 8313 surprises | same direction 91.2% | same magnitude 69.0%


direction,mixed,negative,positive,unclear,All
direction_extraction,,,,,
mixed,433,275,207,20,935
negative,48,2287,14,10,2359
positive,45,18,4810,14,4887
unclear,24,23,30,55,132
All,550,2603,5061,99,8313


magnitude,large,moderate,none,small,All
magnitude_extraction,,,,,
large,689,698,11,29,1427
moderate,149,3535,47,1388,5119
small,4,213,42,1508,1767
All,842,4446,100,2925,8313


sentiment,cautious,neutral,null,upbeat,All
direction,,,,,
mixed,60,314,22,154,550
negative,1152,1117,126,208,2603
positive,86,1566,223,3186,5061
unclear,5,56,29,9,99
All,1303,3053,400,3557,8313


In [15]:
# Where direction changed: read a few to judge which label is right.
changed = labelled[labelled["direction"] != labelled["direction_extraction"]]
print(f"{len(changed)} surprises changed direction")
for row in changed.sample(min(8, len(changed)), random_state=0).itertuples():
    print(f"[{row.direction_extraction} -> {row.direction}] {row.description}\n")

728 surprises changed direction
[mixed -> positive] Cash generation reversed the weak conversion trend highlighted in the preview: operating cash flow rose 143% and free cash flow rose 156%. The improvement was substantially driven by collecting a large receivables balance, so the surge was not solely evidence of a durable change in cash conversion.

[mixed -> negative] The company disclosed that it will reinvest all $14.4M of tariff refunds recognized this year rather than allow the proceeds to flow through to the bottom line. The preview discussed tariff-related margin relief but did not anticipate this specific use of the refund proceeds.

[mixed -> negative] Gross margin improved from Q1’s negative 302% to negative 271%, but remained well short of the preview’s cited marker of moving toward negative 200% or better. The gross loss was still approximately $8.8 million.

[mixed -> negative] The call disclosed that Q3 EBITDA comparisons will lose roughly $9.5 million of prior-year liqu

## 4. Event features under both label sets

Same events (those whose every surprise has a new label), same feature code. `old` uses the extraction's labels; `new` uses the separate step's labels plus sentiment features.

In [16]:
old_surprises = surprises[surprises["event_id"].isin(labelled_ids)]
feat_old = features.build_event_features(old_surprises, split, events_by_id, tax)
feat_new = features.build_event_features(labelled, split, events_by_id, tax)
keep = lambda f: f[f["event_id"].isin(labelled_ids) & (f["split"] == "train")].reset_index(drop=True)
train_old, train_new = keep(feat_old), keep(feat_new)
assert (train_old["event_id"] == train_new["event_id"]).all()
print(f"{len(train_old)} train events | new-only columns: {[c for c in train_new.columns if c not in train_old.columns][:6]} ...")

1072 train events | new-only columns: ['sent__forward_guidance', 'sent__revenue', 'sent__operating_kpis', 'sent__operating_profitability', 'sent__bottom_line_earnings', 'sent__cash_balance_sheet_liquidity'] ...


## 5. No-fit comparison: single scores

In [17]:
ES = model.ES
y, es = train_new["target_pct"], train_new[ES]
scores = {
    "old net_w": model.rank_to_percentile(train_old["net_w"]),
    "old net_w_first3": model.rank_to_percentile(train_old["net_w_first3"]),
    "new net_w": model.rank_to_percentile(train_new["net_w"]),
    "new net_w_first3": model.rank_to_percentile(train_new["net_w_first3"]),
    "new net_sentiment": model.rank_to_percentile(train_new["net_sentiment"]),
}
display(metrics.evaluate(y, scores, es).round(4))
samples = metrics.paired_bootstrap(y, scores, es, n_boot=2000, seed=0)
metrics.compare_to(samples, "new net_w_first3").round(4)

,n,r2_base,r2_full,delta_r2,spearman
old net_w,1024.0,0.0731,0.2385,0.1654,0.4650
old net_w_first3,1024.0,0.0731,0.2335,0.1604,0.4633
new net_w_first3,1024.0,0.0731,0.2264,0.1534,0.4562
new net_w,1024.0,0.0731,0.2264,0.1533,0.4494
new net_sentiment,1024.0,0.0731,0.1694,0.0963,0.3338


,mean_diff,ci_low,ci_high,P(new net_w_first3 better)
old net_w,-0.0119,-0.0408,0.0156,0.2075
old net_w_first3,-0.0071,-0.0188,0.0044,0.1170
new net_w,0.0003,-0.0257,0.0252,0.5115
new net_sentiment,0.0576,0.0178,0.0960,0.9990


## 6. Ridge cross-validation under both label sets

The same grid and folds as `ridge_model.ipynb`, run on the same events under each label set, chosen by pooled out-of-fold ΔR². The best configuration of each is then compared on its out-of-fold predictions with a paired bootstrap.

In [18]:
alphas = np.logspace(-1, 4, 11)
grid_old = model.grid_search(train_old, preprocess.feature_sets(train_old.columns), alphas)
grid_new = model.grid_search(train_new, preprocess.feature_sets(train_new.columns), alphas)
cols = ["feature_set", "alpha", "target_mode", "log_counts", "cv_delta_r2_mean", "oof_delta_r2", "oof_spearman"]
display(grid_old[cols].head(5).round(4))
grid_new[cols].head(5).round(4)

,feature_set,alpha,target_mode,log_counts,cv_delta_r2_mean,oof_delta_r2,oof_spearman
0,all,1000.0000,raw,False,0.1982,0.1793,0.4871
1,all,1000.0000,residual,False,0.1953,0.1777,0.4767
2,all,3162.2777,residual,False,0.2053,0.1761,0.4857
3,all,1000.0000,raw,True,0.1933,0.1743,0.4783
4,all,3162.2777,raw,False,0.2050,0.1742,0.4887


,feature_set,alpha,target_mode,log_counts,cv_delta_r2_mean,oof_delta_r2,oof_spearman
0,all_sent,1000.0000,raw,False,0.1933,0.1766,0.4884
1,all_sent,1000.0000,residual,False,0.1918,0.1752,0.4783
2,all_sent,3162.2777,residual,False,0.2037,0.1750,0.4853
3,all_sent,3162.2777,residual,True,0.2017,0.1741,0.4775
4,all_sent,1000.0000,residual,True,0.1899,0.1736,0.4707


In [19]:
def best_oof(train, grid):
    best = grid.iloc[0]
    out, _ = model.out_of_fold(train, preprocess.feature_sets(train.columns)[best["feature_set"]], best["alpha"],
                               target_mode=best["target_mode"], log_counts=bool(best["log_counts"]))
    return out.set_index("event_id")

oof_old, oof_new = best_oof(train_old, grid_old), best_oof(train_new, grid_new)
scored = oof_old["oof"].notna()
ids = oof_old.index[scored]
preds = {"ridge old labels": oof_old.loc[ids, "oof"].to_numpy(), "ridge new labels": oof_new.loc[ids, "oof"].to_numpy()}
display(metrics.evaluate(oof_old.loc[ids, "target_pct"], preds, oof_old.loc[ids, ES]).round(4))
samples = metrics.paired_bootstrap(oof_old.loc[ids, "target_pct"], preds, oof_old.loc[ids, ES], n_boot=2000, seed=0)
metrics.compare_to(samples, "ridge new labels").round(4)

,n,r2_base,r2_full,delta_r2,spearman
ridge old labels,843.0,0.0768,0.2562,0.1793,0.4871
ridge new labels,843.0,0.0768,0.2534,0.1766,0.4884


,mean_diff,ci_low,ci_high,P(ridge new labels better)
ridge old labels,-0.0025,-0.016,0.0112,0.3545


### Reading the result

- `P(ridge new labels better)` ≥ 0.9 and a positive mean difference: the separate step helps; label the test set the same way and check it once.
- Around 0.5: no measurable gain; keep the extraction's labels (one call fewer per event). Sentiment may still be worth keeping if `new net_sentiment` adds ΔR² on its own in section 5.
- Below 0.1: the separate step is worse; look at the changed-direction examples in section 3 to see why.